# Cellular Network Auto-Triage — Failure Prediction & Anomaly Detection on LTE/5G Logs

Parses real per-second LTE/5G measurement logs, predicts upcoming throughput drops, forecasts throughput,
flags anomalies, categorizes failures, and auto-generates a report.

**Dataset:** UCC 5G dataset — Raca et al., *Beyond Throughput, The Next Generation: A 5G Dataset with Channel and Context Metrics*, ACM MMSys 2020 ([github.com/uccmisl/5Gdataset](https://github.com/uccmisl/5Gdataset)).

## 0. Get the data (run once)

In [ ]:
import os, subprocess
DATA_ROOT = os.environ.get("DATA_ROOT", "data/5Gdataset")
if not os.path.exists(DATA_ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/uccmisl/5Gdataset.git", DATA_ROOT], check=True)
print("Using data from:", DATA_ROOT)

## 1. Parse the raw measurement logs

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import nettriage as nt
raw = nt.load_traces(DATA_ROOT)
print(f"{len(raw):,} rows from {raw['trace_id'].nunique()} traces")
raw[["source_file", "mobility", "RSRP", "RSRQ", "SNR", "CQI", "DL_bitrate", "CellID"]].head()

In [ ]:
df = nt.clean(raw)
print(f"{len(df):,} usable rows after cleaning")
df[["RSRP", "RSRQ", "SNR", "CQI", "Speed", "DL_bitrate"]].describe().round(1)

## 2. Exploratory analysis

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
df["RSRP"].plot.hist(bins=50, ax=axs[0], title="RSRP (dBm)")
df["SNR"].plot.hist(bins=50, ax=axs[1], title="SNR (dB)")
axs[2].scatter(df["SNR"], df["DL_bitrate"] / 1000, s=2, alpha=0.2); axs[2].set(title="SNR vs DL throughput", xlabel="SNR dB", ylabel="Mbps")
plt.tight_layout(); plt.show()
df.groupby("mobility")[["RSRP", "SNR", "DL_bitrate"]].median().round(1)

## 3. Time-series features + failure labels
For every second we compute rolling-window stats (mean / std / min / slope) of RSRP, RSRQ, SNR, CQI and recent handovers (CellID changes).

**Failure label:** the mean throughput over the next 5 s falls below 50% of the recent 10 s median.

In [ ]:
f = nt.build_features(df)
cols = nt.feature_columns(f)
print(f"{len(f):,} samples, {len(cols)} features, drop rate = {f['drop_next'].mean():.3f}")

## 4. Supervised failure prediction — RandomForest + GridSearchCV
Split by **trace** so the test set contains only unseen drives/sessions (no leakage).

In [ ]:
clf, cls_m, imp, ev = nt.train_failure_classifier(f, cols)
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in cls_m.items() if k not in ("report",)}

In [ ]:
imp.head(12)[::-1].plot.barh(figsize=(7, 4)); plt.title("Feature importance"); plt.tight_layout(); plt.show()

## 5. Throughput forecasting — regression

In [ ]:
reg, reg_m = nt.train_throughput_regressor(f, cols)
reg_m

## 6. Unsupervised anomaly detection — IsolationForest

In [ ]:
flag, an_m = nt.detect_anomalies(f)
an_m

## 7. Failure categorization — KMeans on drop events

In [ ]:
events = nt.categorize_failures(f)
events["category"].value_counts()

## 8. Auto-generate the report

In [ ]:
summary = nt.write_report("output", f, cls_m, imp, ev, reg_m, flag, an_m, events)
print(open("output/report.md").read()[:1500])

## 9. Resume-ready numbers
Copy these into the resume bullets. They come straight from this run.

In [ ]:
d, c, r, a = summary["dataset"], summary["failure_prediction"], summary["throughput_regression"], summary["anomaly_detection"]
print(f"- Automated failure triage, as measured by {c['f1']:.2f} F1 / {c['roc_auc']:.2f} ROC-AUC predicting throughput drops 5 s ahead on {c['test_traces']} unseen LTE/5G traces "
      f"(vs {c['rule_baseline_f1']:.2f} F1 for a signal-threshold rule), by parsing {d['rows']:,} network log samples and training a supervised RandomForest in scikit-learn with hyperparameter tuning (GridSearchCV).")
print(f"- Detected network degradations early, as measured by {a['lift']:.1f}x lift in failure rate among flagged samples ({a['recall_of_drops']:.0%} of drops caught at {a['contamination']:.0%} alert rate), "
      f"by building unsupervised anomaly detection (IsolationForest) on rolling-window time-series features of RSRP, SNR, and CQI in Pandas.")
print(f"- Forecast downlink throughput, as measured by {r['mae_mbps']:.2f} Mbps MAE (vs {r['naive_mae_mbps']:.2f} Mbps naive baseline), by training a gradient-boosted regression model on wireless signal metrics.")
print(f"- Generated automated insights, as measured by a one-command report categorizing {d['drop_events']:,} failure events into coverage, interference, congestion, and mobility causes, "
      f"by scripting KMeans clustering and Matplotlib report generation in a GitHub Actions CI pipeline.")